In [6]:
import os
import re
import io
import time
from PIL import Image
import hashlib
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image
from tqdm import tqdm

from playwright.async_api import async_playwright
import nest_asyncio
import asyncio

nest_asyncio.apply()

# =========================
# CONFIG
# =========================
INPUT_PATH = "../data/clean/weblink_text.csv"
INPUT_SEP = ";"

OUTPUT_DIR = "../figures.web_photos"
OUTPUT_CSV = "../data/clean/web_photos_manifest.csv"

MAX_IMAGES_PER_PAGE = 3
REQUEST_TIMEOUT = 20
PLAYWRIGHT_TIMEOUT_MS = 25000

MIN_WIDTH = 250
MIN_HEIGHT = 250
MIN_FILE_BYTES = 8_000

SLEEP_BETWEEN_PAGES = 0.2

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

BAD_IMAGE_PATTERNS = re.compile(
    r"logo|icon|sprite|avatar|banner|ads?|thumb|thumbnail|"
    r"placeholder|favicon|social|facebook|instagram|twitter|"
    r"linkedin|youtube|tiktok|pixel|tracking|cookie|brand|"
    r"header|footer|nav|menu",
    flags=re.IGNORECASE
)

GOOD_IMAGE_PATTERNS = re.compile(
    r"object|artifact|art|lot|item|collection|museum|full|large|zoom|main|hero|image",
    flags=re.IGNORECASE
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def clean_text(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()

def normalize_url(base_url, candidate_url):
    if not candidate_url:
        return ""
    return urljoin(base_url, candidate_url.strip())

def parse_srcset(srcset_value):
    if not srcset_value:
        return ""
    parts = [p.strip() for p in srcset_value.split(",") if p.strip()]
    if not parts:
        return ""
    last = parts[-1].split()[0]
    return last.strip()

def image_extension_from_content_type(content_type):
    if not content_type:
        return ""
    content_type = content_type.lower()
    if "jpeg" in content_type or "jpg" in content_type:
        return ".jpg"
    if "png" in content_type:
        return ".png"
    if "webp" in content_type:
        return ".webp"
    if "gif" in content_type:
        return ".gif"
    if "bmp" in content_type:
        return ".bmp"
    if "tiff" in content_type:
        return ".tiff"
    return ""

def image_extension_from_bytes(content):
    kind = imghdr.what(None, h=content)
    if kind == "jpeg":
        return ".jpg"
    if kind:
        return f".{kind}"
    return ""

def score_candidate(url, alt_text="", extra_text="", width=None, height=None, source="img"):
    blob = clean_text(f"{url} {alt_text} {extra_text}")
    score = 0

    if not url:
        return -10000

    if GOOD_IMAGE_PATTERNS.search(blob):
        score += 120
    if BAD_IMAGE_PATTERNS.search(blob):
        score -= 600

    if re.search(r"\.(jpg|jpeg|png|webp)(\?|$)", url, flags=re.I):
        score += 60

    try:
        if width and int(width) >= 400:
            score += 80
        if height and int(height) >= 400:
            score += 80
    except Exception:
        pass

    if source == "og:image":
        score += 300
    elif source == "twitter:image":
        score += 220
    elif source == "playwright_img":
        score += 140
    elif source == "img":
        score += 80

    return score

def extract_candidates_from_soup(page_url, soup, source_label="img"):
    candidates = []

    for meta in soup.find_all("meta"):
        prop = (meta.get("property") or meta.get("name") or "").strip().lower()
        content = (meta.get("content") or "").strip()

        if prop == "og:image" and content:
            candidates.append({
                "url": normalize_url(page_url, content),
                "alt": "",
                "extra": prop,
                "width": None,
                "height": None,
                "source": "og:image"
            })

        elif prop == "twitter:image" and content:
            candidates.append({
                "url": normalize_url(page_url, content),
                "alt": "",
                "extra": prop,
                "width": None,
                "height": None,
                "source": "twitter:image"
            })

    for img in soup.find_all("img"):
        src = (
            img.get("src")
            or img.get("data-src")
            or img.get("data-original")
            or img.get("data-lazy-src")
            or img.get("data-image")
            or ""
        ).strip()

        srcset = (img.get("srcset") or img.get("data-srcset") or "").strip()
        if srcset:
            src = parse_srcset(srcset) or src

        if not src:
            continue

        full = normalize_url(page_url, src)
        alt = clean_text(img.get("alt", ""))
        cls = " ".join(img.get("class", [])) if img.get("class") else ""
        ident = clean_text(img.get("id", ""))
        title = clean_text(img.get("title", ""))
        extra = clean_text(f"{cls} {ident} {title}")
        width = img.get("width")
        height = img.get("height")

        candidates.append({
            "url": full,
            "alt": alt,
            "extra": extra,
            "width": width,
            "height": height,
            "source": source_label
        })

    deduped = []
    seen = set()
    for c in candidates:
        u = c["url"]
        if not u or u in seen:
            continue
        seen.add(u)
        c["score"] = score_candidate(
            c["url"], c["alt"], c["extra"], c["width"], c["height"], c["source"]
        )
        deduped.append(c)

    deduped.sort(key=lambda x: x["score"], reverse=True)
    return deduped

def fetch_image_content(image_url):
    try:
        r = requests.get(image_url, headers=HEADERS, timeout=REQUEST_TIMEOUT, stream=True)
        r.raise_for_status()
        return r.content, r.headers.get("Content-Type", ""), ""
    except Exception as e:
        return None, None, str(e)

def validate_image(content):
    if content is None:
        return False, None, None, "no_content"

    if len(content) < MIN_FILE_BYTES:
        return False, None, None, "too_small_bytes"

    try:
        img = Image.open(io.BytesIO(content))
        width, height = img.size
        if width < MIN_WIDTH or height < MIN_HEIGHT:
            return False, width, height, "too_small_dimensions"
        return True, width, height, "ok"
    except Exception:
        return False, None, None, "invalid_image"

def build_filename(artifact_id, rank, image_url, content, ext):
    url_hash = hashlib.sha1(image_url.encode("utf-8")).hexdigest()[:10]
    content_hash = hashlib.sha1(content).hexdigest()[:12]
    safe_artifact = re.sub(r"[^A-Za-z0-9_-]+", "_", str(artifact_id))
    return f"{safe_artifact}_img{rank}_{url_hash}_{content_hash}{ext}"

def looks_like_bad_image_candidate(candidate):
    blob = clean_text(f"{candidate['url']} {candidate['alt']} {candidate['extra']}")
    return bool(BAD_IMAGE_PATTERNS.search(blob))

# =========================
# EXTRACTION HTML
# =========================
def get_html_requests(page_url):
    try:
        r = requests.get(page_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r.text, ""
    except Exception as e:
        return None, str(e)

# =========================
# EXTRACTION JS VIA PLAYWRIGHT
# =========================
def get_candidates_playwright(page_url, browser):
    page = browser.new_page(user_agent=HEADERS["User-Agent"])
    page.set_default_timeout(PLAYWRIGHT_TIMEOUT_MS)

    try:
        page.goto(page_url, wait_until="networkidle")
        page_html = page.content()
        soup = BeautifulSoup(page_html, "html.parser")
        candidates = extract_candidates_from_soup(page_url, soup, source_label="playwright_img")

        # bonus: images visibles dans le DOM
        dom_imgs = page.locator("img")
        count = min(dom_imgs.count(), 80)

        extra_candidates = []
        for i in range(count):
            try:
                img = dom_imgs.nth(i)
                src = img.get_attribute("src") or img.get_attribute("data-src") or ""
                srcset = img.get_attribute("srcset") or ""
                if srcset:
                    src = parse_srcset(srcset) or src

                if not src:
                    continue

                full = normalize_url(page_url, src)
                alt = clean_text(img.get_attribute("alt") or "")
                cls = clean_text(img.get_attribute("class") or "")
                width = img.get_attribute("width")
                height = img.get_attribute("height")

                extra_candidates.append({
                    "url": full,
                    "alt": alt,
                    "extra": cls,
                    "width": width,
                    "height": height,
                    "source": "playwright_img",
                    "score": score_candidate(full, alt, cls, width, height, "playwright_img")
                })
            except Exception:
                pass

        all_candidates = candidates + extra_candidates

        deduped = []
        seen = set()
        for c in sorted(all_candidates, key=lambda x: x["score"], reverse=True):
            u = c["url"]
            if not u or u in seen:
                continue
            seen.add(u)
            deduped.append(c)

        return deduped, ""
    except Exception as e:
        return [], str(e)
    finally:
        page.close()

# =========================
# DOWNLOAD LOGIC
# =========================
def download_best_images_for_page(artifact_id, page_url, browser=None):
    results = []

    html_text, req_err = get_html_requests(page_url)
    candidates = []

    if html_text:
        soup = BeautifulSoup(html_text, "html.parser")
        candidates = extract_candidates_from_soup(page_url, soup, source_label="img")

    # fallback JS si pas assez de candidats solides
    if (len(candidates) < MAX_IMAGES_PER_PAGE) and browser is not None:
        js_candidates, js_err = get_candidates_playwright(page_url, browser)
        merged = candidates + js_candidates

        deduped = []
        seen = set()
        for c in sorted(merged, key=lambda x: x["score"], reverse=True):
            u = c["url"]
            if not u or u in seen:
                continue
            seen.add(u)
            deduped.append(c)
        candidates = deduped

    if not candidates:
        status = "no_candidates"
        if req_err:
            status = f"page_error: {req_err}"
        results.append({
            "artifact_id": artifact_id,
            "source_page_url": page_url,
            "image_rank": None,
            "image_url": None,
            "local_filename": None,
            "width": None,
            "height": None,
            "status": status
        })
        return results

    kept = 0
    seen_hashes = set()

    for cand in candidates[:40]:
        if kept >= MAX_IMAGES_PER_PAGE:
            break

        if looks_like_bad_image_candidate(cand):
            continue

        image_url = cand["url"]
        content, content_type, img_err = fetch_image_content(image_url)
        if img_err:
            continue

        ok, width, height, validation_status = validate_image(content)
        if not ok:
            continue

        content_hash = hashlib.sha1(content).hexdigest()
        if content_hash in seen_hashes:
            continue
        seen_hashes.add(content_hash)

        ext = image_extension_from_content_type(content_type) or image_extension_from_bytes(content) or ".jpg"
        filename = build_filename(artifact_id, kept + 1, image_url, content, ext)
        filepath = os.path.join(OUTPUT_DIR, filename)

        try:
            with open(filepath, "wb") as f:
                f.write(content)

            kept += 1
            results.append({
                "artifact_id": artifact_id,
                "source_page_url": page_url,
                "image_rank": kept,
                "image_url": image_url,
                "local_filename": filename,
                "width": width,
                "height": height,
                "status": "downloaded"
            })
        except Exception as e:
            results.append({
                "artifact_id": artifact_id,
                "source_page_url": page_url,
                "image_rank": kept + 1,
                "image_url": image_url,
                "local_filename": None,
                "width": width,
                "height": height,
                "status": f"save_error: {e}"
            })

    if kept == 0:
        results.append({
            "artifact_id": artifact_id,
            "source_page_url": page_url,
            "image_rank": None,
            "image_url": None,
            "local_filename": None,
            "width": None,
            "height": None,
            "status": "no_valid_images"
        })

    return results

# =========================
# MAIN
# =========================
df = pd.read_csv(INPUT_PATH, sep=INPUT_SEP, encoding="utf-8-sig")

required_cols = {"artifact_id", "lien_ok"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Colonnes manquantes: {missing}")

work_df = df.loc[
    df["lien_ok"].notna() & (df["lien_ok"].astype(str).str.strip() != ""),
    ["artifact_id", "lien_ok"]
].copy()

all_rows = []


manifest = pd.DataFrame(all_rows)
manifest.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Terminé.")
print("Dossier images :", OUTPUT_DIR)
print("Manifest CSV :", OUTPUT_CSV)
print("\nRésumé status :")
print(manifest["status"].value_counts(dropna=False))
print("\nAperçu :")
print(manifest.head(20))

Terminé.
Dossier images : ../figures.web_photos
Manifest CSV : ../data/clean/web_photos_manifest.csv

Résumé status :


KeyError: 'status'

In [9]:
async def download_best_images_for_page_async(artifact_id, page_url, browser=None):
    results = []

    try:
        html_text, req_err = get_html_requests(page_url)
        candidates = []

        if html_text:
            soup = BeautifulSoup(html_text, "html.parser")
            candidates = extract_candidates_from_soup(page_url, soup, source_label="img")

        if (len(candidates) < MAX_IMAGES_PER_PAGE) and browser is not None:
            js_candidates, _ = await get_candidates_playwright_async(page_url, browser)
            merged = candidates + js_candidates

            deduped = []
            seen = set()
            for c in sorted(merged, key=lambda x: x["score"], reverse=True):
                u = c["url"]
                if not u or u in seen:
                    continue
                seen.add(u)
                deduped.append(c)
            candidates = deduped

        if not candidates:
            return [{
                "artifact_id": artifact_id,
                "source_page_url": page_url,
                "image_rank": None,
                "image_url": None,
                "local_filename": None,
                "width": None,
                "height": None,
                "status": "no_candidates"
            }]

        kept = 0
        seen_hashes = set()

        for cand in candidates[:40]:
            if kept >= MAX_IMAGES_PER_PAGE:
                break

            if looks_like_bad_image_candidate(cand):
                continue

            image_url = cand["url"]
            content, content_type, img_err = fetch_image_content(image_url)
            if img_err:
                continue

            ok, width, height, _ = validate_image(content)
            if not ok:
                continue

            content_hash = hashlib.sha1(content).hexdigest()
            if content_hash in seen_hashes:
                continue
            seen_hashes.add(content_hash)

            ext = image_extension_from_content_type(content_type) or image_extension_from_bytes(content) or ".jpg"
            filename = build_filename(artifact_id, kept + 1, image_url, content, ext)
            filepath = os.path.join(OUTPUT_DIR, filename)

            with open(filepath, "wb") as f:
                f.write(content)

            kept += 1
            results.append({
                "artifact_id": artifact_id,
                "source_page_url": page_url,
                "image_rank": kept,
                "image_url": image_url,
                "local_filename": filename,
                "width": width,
                "height": height,
                "status": "downloaded"
            })

        if kept == 0:
            return [{
                "artifact_id": artifact_id,
                "source_page_url": page_url,
                "image_rank": None,
                "image_url": None,
                "local_filename": None,
                "width": None,
                "height": None,
                "status": "no_valid_images"
            }]

        return results

    except Exception as e:
        return [{
            "artifact_id": artifact_id,
            "source_page_url": page_url,
            "image_rank": None,
            "image_url": None,
            "local_filename": None,
            "width": None,
            "height": None,
            "status": f"error: {e}"
        }]

In [10]:
async def debug_one_row(work_df):
    test_row = work_df.iloc[0]
    artifact_id = test_row["artifact_id"]
    page_url = str(test_row["lien_ok"]).strip()

    print("artifact_id:", artifact_id)
    print("page_url:", page_url)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        rows = await download_best_images_for_page_async(
            artifact_id,
            page_url,
            browser=browser
        )
        await browser.close()

    print(rows)
    return rows

rows = await debug_one_row(work_df)

artifact_id: 0-آثارنا المنهوبة_25_p001_727222a7
page_url: https://www.lot-art.com/auction-lots/South_ArabianYemeni-fragment-finely-carved-stone-wit/526-sloane-25-2.5.0


NotImplementedError: 